# Lesson 5: Obstacle Avoidance

Your mission is to make the car respond to an obstacle using real distance measurements. Build an obstacle stopper, test an escape maneuver, then connect them.

## Get ready

- Select the **Robot Car** kernel and close any other car-control sessions.
- Securely elevate the driven wheels before running new movement code.
- Keep hands, cables, and clothing clear. Have an instructor present and know how to switch off the car safely.
- For ground tests, use an approved flat, open floor with a stable obstacle and ample clear space around the car.

Run cells individually. Movement is disabled initially. Enable only the activity you are testing, then set its flag back to `False`. Do not use Run All with movement enabled.

## 1. Prepare your tools

Run this setup cell. The distance helper is familiar from Lesson 4. The short movement helper stops the motors after each step. Read the code before using it.

In [ ]:
import math
import time

from betabox_robotics import BetaboxCar
from betabox_robotics.runtime.errors import RobotRuntimeError


def read_distance(car):
    try:
        distance = car.distance()
    except RobotRuntimeError as error:
        print("Reading failed:", error)
        return None

    if distance is None or not math.isfinite(distance) or distance <= 0:
        return None
    return distance


def finish_driving(car):
    try:
        car.stop()
        car.center()
    finally:
        car.close()


def move_for(car, direction, seconds=0.1):
    if direction not in ("forward", "backward"):
        raise ValueError("Choose forward or backward.")
    if not 0 < seconds <= 0.5:
        raise ValueError("Use a duration between 0 and 0.5 seconds.")
    try:
        if direction == "forward":
            car.forward(15)
        else:
            car.backward(15)
        time.sleep(seconds)
    finally:
        car.stop()

## 2. Test your decisions without hardware

Complete `choose_action()` so it returns `"sensor error"`, `"obstacle"`, or `"forward"`. Treat `None`, nonfinite numbers and values at or below zero as sensor errors. Below the threshold means obstacle; equality permits a short forward step.

The default threshold is 30 cm. It is a practice setting, not a guaranteed stopping distance. Your instructor may increase it. Run the checks and fix your function until all pass.

In [ ]:
def choose_action(distance, safe_distance):
    # TODO: Check for an invalid measurement first.
    # TODO: Check for an obstacle below safe_distance.
    # TODO: Return "forward" for other usable readings.
    return "sensor error"  # Replace this placeholder with your logic.


def check_decisions():
    cases = [
        (None, 30, "sensor error"),
        (float("nan"), 30, "sensor error"),
        (float("inf"), 30, "sensor error"),
        (-1, 30, "sensor error"),
        (0, 30, "sensor error"),
        (10, 30, "obstacle"),
        (29.9, 30, "obstacle"),
        (30, 30, "forward"),
        (50, 30, "forward"),
        (35, 40, "obstacle"),
    ]
    passed = True
    for distance, threshold, expected in cases:
        actual = choose_action(distance, threshold)
        correct = actual == expected
        print(distance, "->", actual, "|", "PASS" if correct else "TRY AGAIN")
        if not correct:
            passed = False
    return passed


check_decisions()

## 3. Build an Obstacle Stopper

Complete the loop inside `obstacle_stopper()`. Each check should read the distance using `read_distance(car)`, call `choose_action()`, display the result, and return early for an obstacle or sensor error. Only move in the forward branch, using `move_for(car, "forward", 0.1)`.

Keep the ten-check limit and the final `"limit reached"` result. Each movement ends before the next measurement starts. If the tests in Section 2 fail, do not enable hardware.

In [ ]:
def obstacle_stopper(car, safe_distance=30):
    if not math.isfinite(safe_distance) or safe_distance < 30:
        raise ValueError("Use an instructor-approved threshold of at least 30 cm.")
    car.stop()
    car.center()

    for check in range(10):
        # TODO: Read the distance and choose an action.
        # TODO: Print the measurement and action.
        # TODO: Return "sensor error" or "obstacle" when appropriate.
        # TODO: Only for "forward", make a 0.1-second forward step.
        return "not implemented"  # Remove this placeholder.

    return "limit reached"

## 4. Check your stopper on the stand

After completing Sections 2 and 3, ask your instructor to review your code. Keep the driven wheels elevated. Hold a flat target in front of the sensor, staying clear of moving parts.

Try a nearby target, a farther target, and an unusable reading (if one occurs). Verify that the motors remain stopped whenever the reading is too close or fails. A clear reading should produce only short steps. Ground-test only after instructor approval, with ample stopping room.

In [ ]:
run_stopper = False
safe_distance = 30

if run_stopper and check_decisions():
    car = BetaboxCar()
    try:
        result = obstacle_stopper(car, safe_distance)
        print("Finished:", result)
    finally:
        finish_driving(car)
else:
    print("Stopper disabled or decision tests incomplete.")

## 5. Plan an escape route

Sketch the approved testing area. Identify the obstacle and confirm the space behind and beside the car is clear. Record your plan here before writing code.

| Movement | Direction or steering angle | Duration (seconds) |
|---|---|---|
| Reverse | | |
| Adjust steering | | |
| Follow curve | | |
| Stop and center | | |

Use speed 15 through `move_for()`. Keep each step at or below 0.5 seconds. Stop before reversing or changing steering. The sensor cannot verify side or rear clearance. A forward reading also does not guarantee that the curved path is clear.

## 6. Test your maneuver independently

Write your planned movements inside `avoid_obstacle()`. Use the short movement helper, steering methods and a brief pause between opposite motor directions. Do not add a repeating loop here.

Start on the stand. After approval, test on an open floor without an obstacle. Adjust one setting at a time.

In [ ]:
def avoid_obstacle(car):
    car.stop()
    # TODO: Center steering and reverse briefly if your approved plan requires it.
    # TODO: Pause with the motors stopped.
    # TODO: Set your planned steering angle and perform a short forward step.
    # TODO: Stop and center the steering.
    pass


run_maneuver = False

if run_maneuver:
    car = BetaboxCar()
    try:
        avoid_obstacle(car)
    finally:
        finish_driving(car)
else:
    print("Maneuver disabled. Complete your plan and obtain approval first.")

## 7. Final challenge: Obstacle Avoidance

Combine your two functions. Complete the condition so that `avoid_obstacle()` runs only for the `"obstacle"` result from your stopper. Never run the maneuver for a sensor error, an unfinished program, or a check limit.

Before enabling this test, return the earlier movement flags to `False`. Ask your instructor to approve the complete code and confirm the escape route is clear. Test with wheels elevated first, then on the approved floor. If the area changes, stop testing and reassess it.

In [ ]:
run_project = False
escape_route_approved = False

if run_project and escape_route_approved and check_decisions():
    car = BetaboxCar()
    try:
        result = obstacle_stopper(car, safe_distance)
        car.stop()
        print("Stopper result:", result)
        # TODO: Only if the result is "obstacle", call your avoidance function.
    finally:
        finish_driving(car)
else:
    print("Project disabled. Complete the code, tests and clearance checks first.")

## 8. Test and improve

Record the results of your supervised tests. For sensor failure, use the simulated decision test if a real failed reading cannot be produced reliably. Do not disconnect hardware or create an unsafe condition to force a failure.

| Test | Expected behavior | Actual result / change needed |
|---|---|---|
| Initially too close (stopper only) | No forward movement | |
| Valid clear reading (stopper only) | Short forward steps | |
| Obstacle detected (combined project) | Stop, then approved maneuver | |
| Invalid reading | Stop, no avoidance | |
| Ten clear checks | Stop, no avoidance | |

If a ground test is unsuccessful, return to the stand. Change one setting at a time and repeat your tests from the same starting position.

## Finish up

Confirm the motors are stopped and all three movement flags are `False`. If a movement or cleanup fails, ask your instructor for assistance and safely switch off the car if needed. Save your work.

Return to Lesson 5 on the Betabox portal to complete your reflection.